# 🐟 Dried Fish Quality Classification Pipeline
### ResNet-18 · PyTorch · Google Colab

**Classes:** `High_Quality` | `Medium_Quality` | `Low_Quality`

**Dataset structure expected:**
```
/content/dataset/
  train/  High_Quality/  Medium_Quality/  Low_Quality/
  valid/  High_Quality/  Medium_Quality/  Low_Quality/
  test/   High_Quality/  Medium_Quality/  Low_Quality/
```

> **Runtime → Change runtime type → T4 GPU** before running!

---
| Step | Section |
|------|---------|
| 1 | Install & Imports |
| 2 | Global Config |
| 3 | Data Loading & Transforms |
| 4 | Exploratory Data Analysis (EDA) |
| 5 | DataLoaders |
| 6 | Model Setup (ResNet-18) |
| 7 | Training |
| 8 | Evaluation |
| 9 | Training Curves |
| 10 | Inference & Prediction Function |
| 11 | Save & Load Model |
| 12 | Bonus: Improvement Tips |


##  Step 1 — Install Dependencies

In [ ]:
# Run this cell first — installs any missing packages
!pip install -q torch torchvision scikit-learn matplotlib seaborn Pillow

import importlib, sys
for pkg in ["torch", "torchvision", "sklearn", "matplotlib", "seaborn", "PIL"]:
    v = importlib.import_module(pkg).__version__ if pkg != "PIL" else importlib.import_module("PIL").__version__
    print(f"  ✅ {pkg:<15} {v}")


##  Imports

In [ ]:
import os, copy, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_fscore_support, accuracy_score,
)
from PIL import Image

print("✅ All imports successful")


##  Step 2 — Global Configuration
> **Edit values here** — they flow through the entire pipeline.

In [ ]:
# ─── EDIT THESE AS NEEDED ────────────────────────────────────────────────────
CONFIG = {
    "data_root":        "/content/dataset",   # ← path to your dataset
    "batch_size":       32,
    "num_epochs":       20,
    "learning_rate":    1e-4,
    "num_classes":      3,
    "img_size":         224,                  # ResNet standard input
    "num_workers":      2,
    "uncertainty_thr":  0.60,                 # below this → "UNCERTAIN"
    "model_save_path":  "dried_fish_resnet18.pth",
    "seed":             42,
}

# Reproducibility
torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

# Device (auto-selects GPU if available)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"   GPU    : {torch.cuda.get_device_name(0)}")

# ImageNet normalisation constants (required for pretrained ResNet)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

CLASS_COLORS = {
    "High_Quality":   "#2ecc71",
    "Medium_Quality": "#f39c12",
    "Low_Quality":    "#e74c3c",
}


##  Step 3 — Data Loading & Transforms

* **Training** → augmentation (flip, rotate, colour jitter) + normalise  
* **Valid / Test** → resize + normalise only (no augmentation)


In [ ]:
def build_transforms():
    train_tf = transforms.Compose([
        transforms.Resize((CONFIG["img_size"] + 32, CONFIG["img_size"] + 32)),
        transforms.RandomCrop(CONFIG["img_size"]),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.3, contrast=0.3,
                               saturation=0.2, hue=0.05),
        transforms.RandomGrayscale(p=0.05),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    return train_tf, eval_tf


def load_datasets(data_root, train_tf, eval_tf):
    paths = {
        "train": os.path.join(data_root, "train"),
        "valid": os.path.join(data_root, "valid"),
        "test":  os.path.join(data_root, "test"),
    }
    for split, p in paths.items():
        if not os.path.isdir(p):
            raise FileNotFoundError(f"❌ Missing folder: {p}")

    dsets = {
        "train": datasets.ImageFolder(paths["train"], transform=train_tf),
        "valid": datasets.ImageFolder(paths["valid"], transform=eval_tf),
        "test":  datasets.ImageFolder(paths["test"],  transform=eval_tf),
    }
    return dsets


# ── Run ───────────────────────────────────────────────────────────────────────
train_tf, eval_tf = build_transforms()
datasets_         = load_datasets(CONFIG["data_root"], train_tf, eval_tf)
CLASS_NAMES       = datasets_["train"].classes

print(f"Classes : {CLASS_NAMES}")
for split, ds in datasets_.items():
    print(f"  {split:<6}  {len(ds):>5} images")


##  Step 4 — Exploratory Data Analysis (EDA)

In [ ]:
def run_eda(datasets_):
    split_counts = {}
    for split, ds in datasets_.items():
        counts = {}
        for cls_idx, cls_name in enumerate(ds.classes):
            counts[cls_name] = sum(1 for _, lbl in ds.samples if lbl == cls_idx)
        split_counts[split] = counts
        total = sum(counts.values())
        print(f"\n  [{split.upper()}]  total: {total}")
        for cls, cnt in counts.items():
            pct = cnt / total * 100
            bar = "█" * int(pct / 5)
            print(f"    {cls:<18}  {cnt:>5} ({pct:5.1f}%)  {bar}")

    # Imbalance check
    train_counts = list(split_counts["train"].values())
    ratio = max(train_counts) / (min(train_counts) + 1e-9)
    print(f"\n  Imbalance ratio (max/min): {ratio:.2f}x  ", end="")
    if ratio > 3:
        print("⚠️  HIGH — consider WeightedRandomSampler")
    elif ratio > 1.5:
        print("ℹ️  MODERATE — watch per-class F1")
    else:
        print("✅  Balanced")

    # Bar chart
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle("Class Distribution per Split", fontsize=14, fontweight="bold")
    for ax, (split, counts) in zip(axes, split_counts.items()):
        bars = ax.bar(
            counts.keys(), counts.values(),
            color=[CLASS_COLORS.get(c, "#888") for c in counts.keys()],
            edgecolor="black", linewidth=0.7
        )
        ax.set_title(split.upper(), fontsize=12)
        ax.set_ylabel("Image Count")
        ax.tick_params(axis="x", rotation=20)
        for bar, val in zip(bars, counts.values()):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.5, str(val),
                    ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    plt.savefig("class_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()
    return split_counts


split_counts = run_eda(datasets_)


###  Sample Training Images

In [ ]:
def show_sample_images(dataset, n_per_class=4):
    unnorm = transforms.Normalize(
        mean=[-m / s for m, s in zip(IMAGENET_MEAN, IMAGENET_STD)],
        std=[1 / s for s in IMAGENET_STD]
    )
    n_cls = len(dataset.classes)
    fig, axes = plt.subplots(n_cls, n_per_class,
                             figsize=(n_per_class * 3, n_cls * 3))
    fig.suptitle("Sample Training Images per Class", fontsize=13, fontweight="bold")

    for cls_idx, cls_name in enumerate(dataset.classes):
        cls_paths = [p for p, lbl in dataset.samples if lbl == cls_idx]
        chosen    = np.random.choice(cls_paths,
                                     size=min(n_per_class, len(cls_paths)),
                                     replace=False)
        for col, path in enumerate(chosen):
            img    = dataset.transform(Image.open(path).convert("RGB"))
            img_np = unnorm(img).permute(1, 2, 0).clamp(0, 1).numpy()
            axes[cls_idx][col].imshow(img_np)
            axes[cls_idx][col].axis("off")
            if col == 0:
                axes[cls_idx][col].set_ylabel(
                    cls_name, fontsize=10, fontweight="bold",
                    color=CLASS_COLORS.get(cls_name, "#000")
                )
    plt.tight_layout()
    plt.savefig("sample_images.png", dpi=150, bbox_inches="tight")
    plt.show()


show_sample_images(datasets_["train"])


##  Step 5 — DataLoaders

In [ ]:
def build_loaders(datasets_):
    loaders = {
        "train": DataLoader(
            datasets_["train"],
            batch_size=CONFIG["batch_size"],
            shuffle=True,                       # ← shuffle ONLY for training
            num_workers=CONFIG["num_workers"],
            pin_memory=(DEVICE.type == "cuda"),
        ),
        "valid": DataLoader(
            datasets_["valid"],
            batch_size=CONFIG["batch_size"],
            shuffle=False,
            num_workers=CONFIG["num_workers"],
            pin_memory=(DEVICE.type == "cuda"),
        ),
        "test": DataLoader(
            datasets_["test"],
            batch_size=CONFIG["batch_size"],
            shuffle=False,
            num_workers=CONFIG["num_workers"],
            pin_memory=(DEVICE.type == "cuda"),
        ),
    }
    imgs, lbls = next(iter(loaders["train"]))
    print(f"✅ DataLoaders ready")
    print(f"   Batch images shape : {tuple(imgs.shape)}")
    print(f"   Batch labels shape : {tuple(lbls.shape)}")
    return loaders


loaders = build_loaders(datasets_)


##  Step 6 — Model Setup (ResNet-18)

The backbone is **frozen** (pretrained ImageNet weights preserved).  
Only the custom classification head is trained — fast & avoids overfitting on small datasets.


In [ ]:
def build_model():
    # Load pretrained ResNet-18
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    # Freeze all backbone layers
    for param in model.parameters():
        param.requires_grad = False

    # Replace the final FC with a custom head
    in_features = model.fc.in_features          # 512 for ResNet-18
    model.fc = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(256, CONFIG["num_classes"]),  # 3 output classes
    )

    model = model.to(DEVICE)

    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"✅ ResNet-18 loaded")
    print(f"   Total params     : {total:,}")
    print(f"   Trainable params : {trainable:,}  (head only)")
    print(f"   Architecture     : FC({in_features}) → Dropout → Linear(256) → ReLU → Linear({CONFIG['num_classes']})")
    return model


model = build_model()


##  Step 7 — Training

* **Loss:** CrossEntropyLoss with automatic class weights (handles imbalance)  
* **Optimiser:** Adam  
* **Scheduler:** ReduceLROnPlateau — halves LR when val loss plateaus  
* **Checkpointing:** best model (highest val accuracy) is restored at the end


In [ ]:
def train_model(model, loaders, datasets_):
    # Compute class weights from training label frequency
    train_labels      = [lbl for _, lbl in datasets_["train"].samples]
    class_sample_count = np.bincount(train_labels)
    weight            = 1.0 / (class_sample_count + 1e-9)
    weight_tensor     = torch.tensor(weight, dtype=torch.float).to(DEVICE)

    criterion = nn.CrossEntropyLoss(weight=weight_tensor)
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=CONFIG["learning_rate"],
        weight_decay=1e-4,
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3, verbose=True
    )

    history = {"train_loss": [], "val_loss": [],
               "train_acc":  [], "val_acc":  []}
    best_val_acc = 0.0
    best_weights = copy.deepcopy(model.state_dict())

    print(f"{'Epoch':>5}  {'Tr-Loss':>8}  {'Tr-Acc':>7}  {'Val-Loss':>9}  {'Val-Acc':>8}  {'LR':>9}")
    print("-" * 56)

    for epoch in range(1, CONFIG["num_epochs"] + 1):
        # ── Train phase ───────────────────────────────────────────────────────
        model.train()
        tr_loss, tr_correct, tr_total = 0.0, 0, 0
        for images, labels in loaders["train"]:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            tr_loss    += loss.item() * images.size(0)
            preds       = model(images).detach().argmax(dim=1)
            tr_correct += (preds == labels).sum().item()
            tr_total   += images.size(0)

        # ── Validation phase ─────────────────────────────────────────────────
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in loaders["valid"]:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                out       = model(images)
                val_loss += criterion(out, labels).item() * images.size(0)
                val_correct += (out.argmax(1) == labels).sum().item()
                val_total   += images.size(0)

        etr_loss = tr_loss  / tr_total
        eval_loss = val_loss / val_total
        etr_acc   = tr_correct  / tr_total
        eval_acc  = val_correct / val_total
        cur_lr    = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(etr_loss)
        history["val_loss"].append(eval_loss)
        history["train_acc"].append(etr_acc)
        history["val_acc"].append(eval_acc)
        scheduler.step(eval_loss)

        flag = " ← best ✅" if eval_acc > best_val_acc else ""
        if eval_acc > best_val_acc:
            best_val_acc = eval_acc
            best_weights = copy.deepcopy(model.state_dict())

        print(f"{epoch:>5}  {etr_loss:>8.4f}  {etr_acc:>6.2%}  "
              f"{eval_loss:>9.4f}  {eval_acc:>7.2%}  {cur_lr:>9.2e}{flag}")

    model.load_state_dict(best_weights)   # restore best checkpoint
    print(f"\n✅ Training complete — best val accuracy: {best_val_acc:.2%}")
    return model, history


model, history = train_model(model, loaders, datasets_)


##  Step 8 — Evaluation on Test Set

In [ ]:
def evaluate_model(model, loaders, class_names):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loaders["test"]:
            images = images.to(DEVICE)
            preds  = model(images).argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    acc  = accuracy_score(all_labels, all_preds)
    prec, rec, f1, sup = precision_recall_fscore_support(
        all_labels, all_preds, average=None, labels=range(len(class_names))
    )

    print("=" * 58)
    print("  TEST SET RESULTS")
    print("=" * 58)
    print(f"  Overall Accuracy : {acc:.4f}  ({acc:.2%})\n")
    print(f"  {'Class':<18}  {'Prec':>6}  {'Recall':>6}  {'F1':>6}  {'N':>6}")
    print("  " + "-" * 46)
    for i, cls in enumerate(class_names):
        print(f"  {cls:<18}  {prec[i]:.4f}  {rec[i]:.4f}  {f1[i]:.4f}  {sup[i]:>6}")
    print(f"\n  Macro F1 : {f1.mean():.4f}")
    print("\n" + classification_report(all_labels, all_preds,
                                        target_names=class_names, digits=4))
    return all_preds, all_labels, acc, prec, rec, f1


all_preds, all_labels, acc, prec, rec, f1 = evaluate_model(model, loaders, CLASS_NAMES)


###  Confusion Matrix

In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names):
    cm     = confusion_matrix(y_true, y_pred)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Confusion Matrix", fontsize=14, fontweight="bold")

    for ax, data, fmt, title in zip(
        axes,
        [cm, cm_pct],
        ["d", ".2%"],
        ["Raw Counts", "Normalised (row %)"]
    ):
        sns.heatmap(data, annot=True, fmt=fmt, cmap="Blues",
                    xticklabels=class_names, yticklabels=class_names,
                    linewidths=0.5, ax=ax)
        ax.set_title(title)
        ax.set_xlabel("Predicted", fontsize=10)
        ax.set_ylabel("True",      fontsize=10)
        ax.tick_params(axis="x", rotation=30)
        ax.tick_params(axis="y", rotation=0)

    plt.tight_layout()
    plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved → confusion_matrix.png")


plot_confusion_matrix(all_labels, all_preds, CLASS_NAMES)


##  Step 9 — Training Curves

In [ ]:
def plot_training_curves(history):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Training History", fontsize=14, fontweight="bold")

    # Loss
    ax1.plot(epochs, history["train_loss"], "b-o", markersize=4, label="Train")
    ax1.plot(epochs, history["val_loss"],   "r-o", markersize=4, label="Validation")
    ax1.set_title("Loss Curve")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Cross-Entropy Loss")
    ax1.legend(); ax1.grid(True, alpha=0.3)

    # Accuracy
    ax2.plot(epochs, [v * 100 for v in history["train_acc"]],
             "b-o", markersize=4, label="Train")
    ax2.plot(epochs, [v * 100 for v in history["val_acc"]],
             "r-o", markersize=4, label="Validation")
    ax2.set_title("Accuracy Curve")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy (%)")
    ax2.legend(); ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved → training_curves.png")


plot_training_curves(history)


##  Step 10 — Inference & Prediction Function

* Converts logits → probabilities via **softmax**  
* Returns `"UNCERTAIN"` if max confidence < threshold (default 0.60)


In [ ]:
# ── Inference transform (no augmentation) ─────────────────────────────────────
_infer_tf = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def predict_image(image_path: str,
                  model=model,
                  class_names=CLASS_NAMES,
                  uncertainty_thr=CONFIG["uncertainty_thr"],
                  device=DEVICE) -> dict:
    """
    Predict quality class for a single fish image.

    Returns:
        {
          "class":         "High_Quality",   # or "UNCERTAIN"
          "confidence":    0.87,
          "probabilities": {"High_Quality": 0.87, "Medium_Quality": 0.09, "Low_Quality": 0.04},
          "status":        "OK",             # or "UNCERTAIN"
          "raw_class":     "High_Quality"    # always the argmax class
        }
    """
    try:
        image = Image.open(image_path).convert("RGB")
    except FileNotFoundError:
        return {"error": f"Image not found: {image_path}"}

    tensor = _infer_tf(image).unsqueeze(0).to(device)   # [1, C, H, W]

    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1).squeeze()  # [num_classes]

    confidence, pred_idx = probs.max(dim=0)
    confidence = confidence.item()
    pred_class = class_names[pred_idx.item()]
    all_probs  = {cls: round(p.item(), 4) for cls, p in zip(class_names, probs)}
    status     = "OK" if confidence >= uncertainty_thr else "UNCERTAIN"

    if status == "UNCERTAIN":
        print(f"⚠️  Low confidence ({confidence:.2%}) — flagged as UNCERTAIN")

    return {
        "class":         pred_class if status == "OK" else "UNCERTAIN",
        "confidence":    round(confidence, 4),
        "probabilities": all_probs,
        "status":        status,
        "raw_class":     pred_class,
    }


print("✅ predict_image() is ready")
print("   Usage:  result = predict_image('/path/to/fish.jpg')")


###  Run Inference on a Test Image

In [ ]:
# ── Auto-pick the first available test image ──────────────────────────────────
demo_path = None
for cls in CLASS_NAMES:
    folder = os.path.join(CONFIG["data_root"], "test", cls)
    if os.path.isdir(folder):
        files = [f for f in os.listdir(folder)
                 if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        if files:
            demo_path = os.path.join(folder, files[0])
            break

if demo_path:
    print(f"Running inference on: {demo_path}\n")
    result = predict_image(demo_path)
    print(json.dumps(result, indent=2))

    # Show the image alongside prediction
    img = Image.open(demo_path).convert("RGB")
    color = CLASS_COLORS.get(result["raw_class"], "#888")
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(
        f"Predicted: {result['class']}\nConfidence: {result['confidence']:.2%}",
        fontsize=12, fontweight="bold", color=color
    )
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  No test images found — update demo_path manually")
    # demo_path = "/content/your_fish_image.jpg"
    # result = predict_image(demo_path)


###  Batch Prediction

In [ ]:
def batch_predict(image_paths: list, **kwargs) -> list:
    """Run predict_image() on a list of image paths."""
    results = []
    for path in image_paths:
        res = predict_image(path, **kwargs)
        res["image_path"] = os.path.basename(path)
        results.append(res)
    return results


# ── Example: predict on all images in one test class ─────────────────────────
test_folder = os.path.join(CONFIG["data_root"], "test", CLASS_NAMES[0])
if os.path.isdir(test_folder):
    sample_paths = [
        os.path.join(test_folder, f)
        for f in os.listdir(test_folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ][:5]                          # limit to 5 for demo

    results = batch_predict(sample_paths)
    print(f"\nBatch Results ({len(results)} images):")
    for r in results:
        status_icon = "✅" if r["status"] == "OK" else "⚠️ "
        print(f"  {status_icon}  {r['image_path']:<35}  "
              f"{r['class']:<18}  conf={r['confidence']:.2%}")


##  Step 11 — Save Model

In [ ]:
def save_model(model, path, class_names, config):
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "class_names":      class_names,
        "config":           config,
        "architecture":     "resnet18",
    }
    torch.save(checkpoint, path)
    size_mb = os.path.getsize(path) / 1e6
    print(f"✅ Model saved → {path}  ({size_mb:.1f} MB)")


save_model(model, CONFIG["model_save_path"], CLASS_NAMES, CONFIG)


###  Load Model for Later Use

In [ ]:
def load_model(path, device=DEVICE):
    """
    Load a saved checkpoint.
    Usage:
        model, class_names, cfg = load_model('dried_fish_resnet18.pth')
        result = predict_image('fish.jpg', model=model, class_names=class_names)
    """
    ckpt        = torch.load(path, map_location=device)
    class_names = ckpt["class_names"]

    base = models.resnet18(weights=None)
    in_f = base.fc.in_features
    base.fc = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_f, 256),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(256, len(class_names)),
    )
    base.load_state_dict(ckpt["model_state_dict"])
    base = base.to(device)
    base.eval()
    print(f"✅ Model loaded from: {path}")
    print(f"   Classes: {class_names}")
    return base, class_names, ckpt["config"]


# ── Test round-trip: save → load → predict ───────────────────────────────────
loaded_model, loaded_classes, loaded_cfg = load_model(CONFIG["model_save_path"])

if demo_path:
    result = predict_image(demo_path, model=loaded_model, class_names=loaded_classes)
    print("\nPrediction from loaded model:")
    print(json.dumps(result, indent=2))


##  Step 12 — Bonus: Improvement Tips

> Optional code snippets — uncomment and run as needed.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TIP 1 — WeightedRandomSampler (better than class_weight alone for imbalance)
# ══════════════════════════════════════════════════════════════════════════════
# from torch.utils.data import WeightedRandomSampler
#
# labels       = [lbl for _, lbl in datasets_["train"].samples]
# class_counts = np.bincount(labels)
# weights      = [1.0 / class_counts[lbl] for lbl in labels]
# sampler      = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
#
# train_loader_balanced = DataLoader(
#     datasets_["train"], batch_size=CONFIG["batch_size"],
#     sampler=sampler, num_workers=CONFIG["num_workers"]
# )


# ══════════════════════════════════════════════════════════════════════════════
# TIP 2 — Unfreeze backbone for full fine-tuning (run after head warmup)
# ══════════════════════════════════════════════════════════════════════════════
# for param in model.parameters():
#     param.requires_grad = True
#
# optimizer_ft = optim.Adam(model.parameters(), lr=1e-5, weight_decay=1e-4)
# scheduler_ft = optim.lr_scheduler.OneCycleLR(
#     optimizer_ft, max_lr=1e-4,
#     steps_per_epoch=len(loaders["train"]),
#     epochs=10
# )
# model, history_ft = train_model(model, loaders, datasets_)


# ══════════════════════════════════════════════════════════════════════════════
# TIP 3 — Stronger augmentations with Albumentations
# ══════════════════════════════════════════════════════════════════════════════
# !pip install -q albumentations
# import albumentations as A
# from albumentations.pytorch import ToTensorV2
#
# albu_train = A.Compose([
#     A.RandomResizedCrop(224, 224, scale=(0.7, 1.0)),
#     A.HorizontalFlip(p=0.5),
#     A.RandomBrightnessContrast(p=0.4),
#     A.CLAHE(p=0.3),
#     A.GaussianBlur(blur_limit=3, p=0.2),
#     A.GridDistortion(p=0.2),
#     A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
#     ToTensorV2(),
# ])


# ══════════════════════════════════════════════════════════════════════════════
# TIP 4 — Stronger backbone (swap resnet18 → efficientnet_b3)
# ══════════════════════════════════════════════════════════════════════════════
# from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
# backbone = efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)
# in_f     = backbone.classifier[1].in_features
# backbone.classifier = nn.Sequential(
#     nn.Dropout(p=0.4),
#     nn.Linear(in_f, CONFIG["num_classes"])
# )
# backbone = backbone.to(DEVICE)


# ══════════════════════════════════════════════════════════════════════════════
# TIP 5 — Optuna hyperparameter search
# ══════════════════════════════════════════════════════════════════════════════
# !pip install -q optuna
# import optuna
#
# def objective(trial):
#     lr      = trial.suggest_float("lr",      1e-5, 1e-3, log=True)
#     dropout = trial.suggest_float("dropout", 0.2,  0.6)
#     wd      = trial.suggest_float("wd",      1e-5, 1e-3, log=True)
#     # build model, train 5 epochs, return val_acc …
#
# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=20)
# print("Best:", study.best_params)


# ══════════════════════════════════════════════════════════════════════════════
# TIP 6 — Test-Time Augmentation (TTA)
# ══════════════════════════════════════════════════════════════════════════════
# def predict_with_tta(image_path, model, class_names, n_tta=5):
#     tta_tf = transforms.Compose([
#         transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
#         transforms.RandomHorizontalFlip(),
#         transforms.ColorJitter(brightness=0.1, contrast=0.1),
#         transforms.ToTensor(),
#         transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
#     ])
#     image  = Image.open(image_path).convert("RGB")
#     probs  = torch.zeros(len(class_names)).to(DEVICE)
#     model.eval()
#     with torch.no_grad():
#         for _ in range(n_tta):
#             t = tta_tf(image).unsqueeze(0).to(DEVICE)
#             probs += torch.softmax(model(t), dim=1).squeeze()
#     probs /= n_tta
#     conf, idx = probs.max(0)
#     return {"class": class_names[idx], "confidence": round(conf.item(), 4)}

print("✅ Tip cells loaded — uncomment any block above and run to apply.")


---
##  Pipeline Complete!

| Artifact | Description |
|---|---|
| `dried_fish_resnet18.pth` | Trained model checkpoint |
| `class_distribution.png` | Class balance bar charts |
| `sample_images.png` | Grid of training samples |
| `confusion_matrix.png` | Raw + normalised confusion matrix |
| `training_curves.png` | Loss & accuracy over epochs |

### Quick-start inference on a new image
```python
model, class_names, cfg = load_model("dried_fish_resnet18.pth")
result = predict_image("/path/to/your_fish.jpg", model=model, class_names=class_names)
print(result)
# → {"class": "High_Quality", "confidence": 0.91, ...}
```
